In [1]:
import pandas as pd

In [2]:
# 1. Đọc dữ liệu từ file .xlsx
df = pd.read_excel("NHÓM_SAN_PHAM.xlsx")

In [3]:
# 2. Tạo cột ngày tạm thời để tính toán
df['temp_date'] = pd.to_datetime(df['year_month'])

In [4]:
# 3. Tính toán đặc tính của từng nhóm sản phẩm
sku_stats = df.groupby('group_name').agg(
    total_months=('year_month', 'nunique'),
    last_sale_month=('temp_date', 'max')
).reset_index()


In [6]:
#  4. Định nghĩa logic phân khúc 5 nhóm (Ngưỡng Inactive < 2024 theo yêu cầu)
def assign_segment_logic(row):
    # Nhóm Inactive: Ngừng phát sinh giao dịch từ trước 01/2024
    if row['last_sale_month'] < pd.Timestamp('2024-01-01'):
        return 'G - Inactive ', 'N/A (Clearance Only)'

    # Các nhóm đang hoạt động (có phát sinh trong 2024 hoặc 2025)
    if row['total_months'] >= 30:
        return 'A - ML Candidate', 'XGBoost, LightGBM, CatBoost, ARIMA, SARIMA, LSTM, Prophet, Ensemble'
    elif row['total_months'] >= 12:
        return 'B - Lightweight ML', 'Random Forest, Holt-Winters, SES, Linear Regression'
    elif row['total_months'] >= 3:
        return 'C - Sparse', 'Moving Average (MA3/MA6), Croston’s Method, Naive'
    else:
        return 'D - Cold Start', 'Naive Forecast, Analogy Method'

# Áp dụng logic để tạo cột phân khúc và mô hình gợi ý
sku_stats[['segment', 'suggested_models']] = sku_stats.apply(
    lambda x: pd.Series(assign_segment_logic(x)), axis=1
)

In [7]:
# 5. Gộp thông tin phân khúc vào dữ liệu gốc (trả year_month về định dạng ban đầu)
final_df = df.merge(sku_stats[['group_name', 'segment', 'suggested_models']], on='group_name', how='left')

In [8]:
# 6. Xuất ra file CSV với đúng 5 cột yêu cầu
output_columns = ['group_name', 'year_month', 'qty', 'segment', 'suggested_models']
final_df[output_columns].to_csv('KET_QUA_PHAN_KHUC_NCKH.csv', index=False, encoding='utf-8-sig')

print("--- THÀNH CÔNG ---")
print("File 'KET_QUA_PHAN_KHUC_NCKH.csv' đã được tạo trong thư mục của bạn.")

--- THÀNH CÔNG ---
File 'KET_QUA_PHAN_KHUC_NCKH.csv' đã được tạo trong thư mục của bạn.
